# Notebook 02 — Feature Engineering
## Machine Learning for Product Gain/Loss Prediction
**Author:** Uzoma Nnaemeka Eze | MSc Data Science | UE Potsdam

---
This notebook constructs all predictive features for both model levels:
- **M2 features** — 9 features from depot-level summary variables
- **D2 tank-level features** — 15 features including Opening Dip (mm), Water Dip, Temperature

**Key design decision:** `Stock_Change` (Closing − Opening Stock) is deliberately excluded  
because it is mathematically equivalent to the target variable `Loss_Gain`, which would  
constitute data leakage and produce artificially perfect model performance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
try:
    ago = pd.read_csv('data/ago_m2.csv', parse_dates=['Date'])
    pms = pd.read_csv('data/pms_m2.csv', parse_dates=['Date'])
    tank_df = pd.read_csv('data/tank_clean.csv', parse_dates=['Date'])
    print(f"M2 AGO: {len(ago)} records | M2 PMS: {len(pms)} records")
    print(f"Tank-level: {len(tank_df)} records")
except Exception as e:
    print(f"Error loading data: {e}")
    ago = pms = tank_df = pd.DataFrame()


## 1. M2 Depot-Level Feature Engineering

In [ ]:
def engineer_m2_features(df):
    """
    Engineer predictive features from M2 daily stock summary.

    Features:
    - Receipt_Delivery_Ratio : Operational balance between inflow and outflow
    - Lag1_Loss              : Previous day Loss/Gain (temporal signal)
    - Lag2_Loss              : Two-day lag Loss/Gain
    - Rolling3_Loss          : 3-day rolling mean of Loss/Gain
    - DayOfWeek              : Day of week (0=Monday, 6=Sunday)
    - Month                  : Calendar month (1-12)
    - Loss_Flag              : Binary target — 1 if Loss_Gain < 0, else 0

    Excluded: Stock_Change — mathematically derives the target (data leakage).
    """
    df = df.copy()
    df['Total_Receipts']   = df['Total_Receipts'].fillna(0)
    df['Total_Deliveries'] = df['Total_Deliveries'].fillna(0)

    df['Receipt_Delivery_Ratio'] = np.where(
        df['Total_Deliveries'] > 0,
        df['Total_Receipts'] / (df['Total_Deliveries'] + 1), 0)

    df['Lag1_Loss']     = df['Loss_Gain'].shift(1).fillna(0)
    df['Lag2_Loss']     = df['Loss_Gain'].shift(2).fillna(0)
    df['Rolling3_Loss'] = df['Loss_Gain'].shift(1).rolling(3, min_periods=1).mean().fillna(0)
    df['DayOfWeek']     = df['Date'].dt.dayofweek
    df['Month']         = df['Date'].dt.month
    df['Loss_Flag']     = (df['Loss_Gain'] < 0).astype(int)
    return df

ago_feat = engineer_m2_features(ago) if len(ago) > 0 else ago
pms_feat = engineer_m2_features(pms) if len(pms) > 0 else pms

M2_FEATURES = ['Opening_Stock', 'Total_Receipts', 'Total_Deliveries',
                'Receipt_Delivery_Ratio', 'Lag1_Loss', 'Lag2_Loss',
                'Rolling3_Loss', 'DayOfWeek', 'Month']

print("M2 Features:", M2_FEATURES)
if len(ago_feat) > 0:
    print(f"\nAGO Loss_Flag distribution:")
    print(ago_feat['Loss_Flag'].value_counts().rename({0:'Gain/Neutral', 1:'Loss'}))
    print(f"\nPMS Loss_Flag distribution:")
    print(pms_feat['Loss_Flag'].value_counts().rename({0:'Gain/Neutral', 1:'Loss'}))


## 2. D2 Tank-Level Feature Engineering

In [ ]:
def engineer_tank_features(df):
    """
    Engineer predictive features from D2 individual tank daily records.
    All temporal features computed within tank groups to prevent cross-tank leakage.
    """
    df = df.sort_values(['Tank', 'Date']).reset_index(drop=True)

    df['Has_Water']    = (df['Water_Dip_mm'] > 0).astype(int)
    df['Dip_Change']   = df.groupby('Tank')['Opening_Dip_mm'].diff().fillna(0)
    df['Lag1_LG']      = df.groupby('Tank')['Loss_Gain'].shift(1).fillna(0)
    df['Lag2_LG']      = df.groupby('Tank')['Loss_Gain'].shift(2).fillna(0)
    df['Rolling3_LG']  = (df.groupby('Tank')['Loss_Gain']
                           .shift(1)
                           .transform(lambda x: x.rolling(3, min_periods=1).mean())
                           .fillna(0))
    df['Has_Receipt']  = (df['Receipt'] > 0).astype(int)
    df['Has_Dispatch'] = (df['Dispatch'] > 0).astype(int)
    df['DayOfWeek']    = df['Date'].dt.dayofweek
    df['Month_Num']    = df['Date'].dt.month
    df['Loss_Flag']    = (df['Loss_Gain'] < 0).astype(int)
    return df

tank_feat = engineer_tank_features(tank_df) if len(tank_df) > 0 else tank_df

TANK_FEATURES = ['Opening_Dip_mm', 'Water_Dip_mm', 'Temperature',
                 'Opening_Vol', 'Receipt', 'Dispatch', 'Has_Water',
                 'Dip_Change', 'Lag1_LG', 'Lag2_LG', 'Rolling3_LG',
                 'Has_Receipt', 'Has_Dispatch', 'DayOfWeek', 'Month_Num']

print(f"Tank Features ({len(TANK_FEATURES)}):", TANK_FEATURES)
if len(tank_feat) > 0:
    for product in ['AGO','PMS']:
        sub = tank_feat[tank_feat['Product']==product]
        print(f"\n{product}: {len(sub)} records | Loss events: {sub['Loss_Flag'].sum()} ({sub['Loss_Flag'].mean()*100:.1f}%)")

# Save engineered features
if len(ago_feat) > 0:
    ago_feat.to_csv('data/ago_feat.csv', index=False)
    pms_feat.to_csv('data/pms_feat.csv', index=False)
    tank_feat.to_csv('data/tank_feat.csv', index=False)
    print("\nFeature datasets saved.")


## 3. Correlation Analysis — M2 Features vs Loss/Gain

In [ ]:
if len(ago_feat) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Feature Correlation with Loss/Gain — AGO and PMS', fontsize=12, fontweight='bold')

    for ax, (product, df_f, color) in zip(axes, [('AGO', ago_feat, '#1A5F9E'), ('PMS', pms_feat, '#C94E2D')]):
        corr = df_f[M2_FEATURES + ['Loss_Gain']].corr()['Loss_Gain'].drop('Loss_Gain').sort_values()
        ax.barh(corr.index, corr.values, color=[color if v > 0 else '#888888' for v in corr.values],
                alpha=0.8, edgecolor='white')
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(f'{product} — Feature Correlation', fontweight='bold', color=color)
        ax.set_xlabel('Pearson Correlation with Loss/Gain')
        ax.grid(axis='x', alpha=0.3)
        ax.set_facecolor('#FAFAFA')

    plt.tight_layout()
    plt.savefig('reports/fig_feature_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()
